# Sign Language Recognition (ASL Alphabet)
End-to-end Machine Learning project on the Kaggle ASL Alphabet dataset.

## STEP 1 - SETUP & IMPORTS
- Import all necessary libraries
- Set random seeds for reproducibility
- Print TensorFlow version

In [10]:
import os
import random
from glob import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2


import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

# Set random seeds for reproducibility
np.random.seed(42)
random.seed(42)
tf.random.set_seed(42)

print('TensorFlow Version:', tf.__version__)

TensorFlow Version: 2.21.0


## STEP 2 - LOAD DATASET
- Load the ASL alphabet dataset from Kaggle directory
- Show the 29 class names
- Display a grid of sample images (5x5) with their labels
- Print total image count per class

In [11]:
# Define Dataset Path
# The dataset on Kaggle is typically found at this location:
DATA_DIR = '/kaggle/input/asl-alphabet/asl_alphabet_train/asl_alphabet_train'

# Fallback path if the nested folder doesn't exist
if not os.path.exists(DATA_DIR):
    DATA_DIR = '/kaggle/input/asl-alphabet/asl_alphabet_train'

# Get all class names
if os.path.exists(DATA_DIR):
    class_names = sorted(os.listdir(DATA_DIR))
    print(f"Total Classes: {len(class_names)}")
    print("Class Names:", class_names)
else:
    print(f"Directory {DATA_DIR} not found. Ensure you are running this on Kaggle with the dataset added.")
    class_names = []

# Collect image paths and their labels
image_paths = []
labels = []

for class_name in class_names:
    class_dir = os.path.join(DATA_DIR, class_name)
    if os.path.isdir(class_dir):
        images = glob(os.path.join(class_dir, '*.jpg'))
        image_paths.extend(images)
        labels.extend([class_name] * len(images))

# Create a DataFrame for easy handling
df = pd.DataFrame({'image_path': image_paths, 'label': labels})

if not df.empty:
    print("\nTotal image count per class:")
    print(df['label'].value_counts().sort_index())

    # Display a 5x5 grid of sample images
    plt.figure(figsize=(12, 12))
    sample_df = df.sample(25, random_state=42).reset_index(drop=True)

    for i in range(25):
        plt.subplot(5, 5, i+1)
        img = cv2.imread(sample_df['image_path'][i])
        if img is not None:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) # BGR to RGB for matplotlib
            plt.imshow(img)
            plt.title(sample_df['label'][i])
        plt.axis('off')
    plt.tight_layout()
    plt.show()

Directory /kaggle/input/asl-alphabet/asl_alphabet_train not found. Ensure you are running this on Kaggle with the dataset added.


## STEP 3 - PREPROCESS
- Resize to 64x64
- Normalize to 0-1
- One-hot encode labels
- Split into 80% train, 10% validation, 10% test
- Data Augmentation using ImageDataGenerator

In [12]:
if not df.empty:
    # We will use ImageDataGenerator which handles resizing, normalization, and one-hot encoding implicitly.
    
    # Train: 80%, Validation: 10%, Test: 10%
    # First, split 80% Train, 20% Temp
    train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
    
    # Next, split Temp into 50% Val, 50% Test (which results in 10% overall each)
    val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])
    
    print(f"Train set size: {len(train_df)}")
    print(f"Validation set size: {len(val_df)}")
    print(f"Test set size: {len(test_df)}")
    
    # Parameters
    IMG_SIZE = (64, 64)
    BATCH_SIZE = 32
    
    # Train generator with Data Augmentation and Normalization
    train_datagen = ImageDataGenerator(
        rescale=1./255,             # Normalize pixel values to 0-1
        rotation_range=15,          # Data Augmentation parameters
        zoom_range=0.1,
        horizontal_flip=True,
        width_shift_range=0.1,
        height_shift_range=0.1
    )
    
    # Validation and Test generators (ONLY Normalization, NO Augmentation)
    val_test_datagen = ImageDataGenerator(rescale=1./255)
    
    print("\nSetting up Data Generators:")
    train_generator = train_datagen.flow_from_dataframe(
        dataframe=train_df,
        x_col='image_path',
        y_col='label',
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical', # Explicitly enables one-hot encoding
        shuffle=True
    )
    
    val_generator = val_test_datagen.flow_from_dataframe(
        dataframe=val_df,
        x_col='image_path',
        y_col='label',
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        shuffle=False
    )
    
    test_generator = val_test_datagen.flow_from_dataframe(
        dataframe=test_df,
        x_col='image_path',
        y_col='label',
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        shuffle=False
    )

## STEP 4 - BUILD CNN MODEL
- Build exact requested architecture using Keras Sequential API
- Compile with Adam (lr=0.001) and categorical_crossentropy

In [13]:
# Build the CNN model architecture
model = Sequential([
    # Block 1
    Conv2D(32, (3, 3), activation='relu', input_shape=(64, 64, 3)),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    
    # Block 2
    Conv2D(64, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    
    # Block 3
    Conv2D(128, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    
    # Flattening
    Flatten(),
    
    # Fully Connected Layers
    Dense(256, activation='relu'),
    Dropout(0.4),
    Dense(128, activation='relu'),
    Dropout(0.3),
    
    # Output Layer for 29 classes
    Dense(29, activation='softmax')
])

# Compile the model
optimizer = keras.optimizers.Adam(learning_rate=0.001)
model.compile(optimizer=optimizer, 
              loss='categorical_crossentropy', 
              metrics=['accuracy'])

# Print Model Summary
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 62, 62, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 62, 62, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 31, 31, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 29, 29, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 29, 29, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 12, 12, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 12, 12, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 6, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 4608)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 256)            │     1,179,904 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 29)             │         3,741 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,310,685 (5.00 MB)

 Trainable params: 1,310,237 (5.00 MB)

 Non-trainable params: 448 (1.75 KB)

## STEP 5 - TRAIN
- Train for 15 epochs with batch size 32
- Use EarlyStopping and ReduceLROnPlateau
- Save best model as `asl_model.h5` using ModelCheckpoint

In [14]:
EPOCHS = 15

# Define Callbacks
early_stopping = EarlyStopping(
    monitor='val_loss', 
    patience=3, 
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', 
    factor=0.5, 
    patience=2, 
    verbose=1
)

model_checkpoint = ModelCheckpoint(
    'asl_model.h5', 
    monitor='val_accuracy', 
    save_best_only=True,
    verbose=1
)

callbacks = [early_stopping, reduce_lr, model_checkpoint]

if not df.empty:
    # Start Training
    history = model.fit(
        train_generator,
        validation_data=val_generator,
        epochs=EPOCHS,
        callbacks=callbacks
    )
else:
    print("Skipping training because dataset is not loaded. Upload notebook to Kaggle to train.")

Skipping training because dataset is not loaded. Upload notebook to Kaggle to train.


## STEP 6 - VISUALIZE RESULTS
- Plot training vs validation accuracy curve
- Plot training vs validation loss curve
- Print final test accuracy

In [15]:
if not df.empty and 'history' in locals():
    # Set up figure
    plt.figure(figsize=(14, 5))
    
    # Plot Accuracy
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
    plt.title('Training vs Validation Accuracy', fontsize=14)
    plt.xlabel('Epochs', fontsize=12)
    plt.ylabel('Accuracy', fontsize=12)
    plt.legend()
    plt.grid(True)
    
    # Plot Loss
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Train Loss', linewidth=2)
    plt.plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
    plt.title('Training vs Validation Loss', fontsize=14)
    plt.xlabel('Epochs', fontsize=12)
    plt.ylabel('Loss', fontsize=12)
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    plt.show()

    # Evaluate on final test set
    print("\nEvaluating model on the Test Set...")
    test_loss, test_accuracy = model.evaluate(test_generator, verbose=1)
    print(f"\nFinal Test Accuracy: {test_accuracy * 100:.2f}%")

## STEP 7 - CONFUSION MATRIX
- Predict on entire test set
- Plot a beautiful confusion matrix using seaborn heatmap
- Print classification report (precision, recall, f1-score per class)

In [16]:
if not df.empty:
    # Ensure generator is reset before predicting to align with true labels
    test_generator.reset()
    
    print("Generating predictions on the test set...")
    predictions = model.predict(test_generator, verbose=1)
    
    # Get predicted classes
    y_pred = np.argmax(predictions, axis=1)
    y_true = test_generator.classes
    
    # Extract class labels mapped by the generator
    class_labels_dict = test_generator.class_indices
    class_labels = list(class_labels_dict.keys())
    
    # Calculate Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    
    # Plot Confusion Matrix
    plt.figure(figsize=(18, 14))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_labels, yticklabels=class_labels)
    plt.title('Confusion Matrix - ASL Alphabet', fontsize=18, fontweight='bold')
    plt.xlabel('Predicted Label', fontsize=14)
    plt.ylabel('True Label', fontsize=14)
    plt.xticks(rotation=45)
    plt.yticks(rotation=0)
    plt.show()
    
    # Print Classification Report
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=class_labels))

## STEP 8 - PREDICTION FUNCTION
- Write a `predict_letter(image_path)` function
- Test it on 5 random test images and show image + prediction side by side

In [17]:
def predict_letter(image_path, model, class_labels):
    '''
    Loads an image from a given path, preprocesses it to match the model's expected 
    input (64x64, normalized), and returns the predicted letter and confidence.
    '''
    # Load image using OpenCV
    img = cv2.imread(image_path)
    if img is None:
        return None, 0.0, None
        
    # Convert BGR (OpenCV default) to RGB
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Resize to match model input shape (64x64)
    img_resized = cv2.resize(img_rgb, (64, 64))
    
    # Normalize pixel values
    img_normalized = img_resized / 255.0
    
    # Expand dimensions to create a batch of size 1: shape (1, 64, 64, 3)
    img_batch = np.expand_dims(img_normalized, axis=0)
    
    # Make prediction
    prediction = model.predict(img_batch, verbose=0)
    predicted_class_idx = np.argmax(prediction[0])
    predicted_letter = class_labels[predicted_class_idx]
    confidence = np.max(prediction[0]) * 100
    
    return predicted_letter, confidence, img_rgb

if not df.empty:
    print("Testing the prediction function on 5 random test images...")
    # Sample 5 random images from the test set
    sample_test_df = test_df.sample(5, random_state=random.randint(0, 10000)).reset_index(drop=True)
    
    plt.figure(figsize=(20, 5))
    for i in range(5):
        img_path = sample_test_df['image_path'][i]
        true_label = sample_test_df['label'][i]
        
        pred_letter, conf, img_rgb = predict_letter(img_path, model, class_labels)
        
        if img_rgb is not None:
            plt.subplot(1, 5, i+1)
            plt.imshow(img_rgb)
            
            # Color title green if correct, red if incorrect
            title_color = 'green' if pred_letter == true_label else 'red'
            plt.title(f"True: {true_label}\nPred: {pred_letter} ({conf:.1f}%)", 
                      color=title_color, fontsize=14, fontweight='bold')
            plt.axis('off')
        
    plt.tight_layout()
    plt.show()

## STEP 9 - SAVE & EXPORT
- Save the final model as `asl_model.h5`
- Save the class labels as `labels.npy`
- Print confirmation

In [18]:
# Save the final model explicitly
model.save('asl_model.h5')

if not df.empty:
    # Save the class labels array to a numpy binary file
    np.save('labels.npy', class_labels)

print("✅ Model successfully saved as 'asl_model.h5'")
print("✅ Class labels successfully saved as 'labels.npy'")
print("🎉 Project is complete and ready for download/submission!")

✅ Model successfully saved as 'asl_model.h5'
✅ Class labels successfully saved as 'labels.npy'
🎉 Project is complete and ready for download/submission!
